# CRNN on log-mel-spectrograms — Strategy A

Trains a CRNN (2 conv blocks + 2-layer BiLSTM, ~1.65M params) on per-segment
log-mel-spectrograms (1 × 128 × 100). Same train/val/test split as
every other model in this project (`splits.npz`, seed=9).

**Goal of this experiment:** test whether explicit temporal modeling
(BiLSTM over conv frames) helps over the frame-pooled CNN.


## Cell 1 — Setup


In [1]:
# ============================================================================
# Cell 1 — Setup: imports, paths, device, reproducibility (CRNN)
# ============================================================================
import sys, time, json
from pathlib import Path

import numpy as np
import torch

# Resolve project root (contains config.json + data/)
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'config.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / '_shared'))

from data       import load_splits
from melspec    import load_cache, MelSpecDataset
from lstm       import CRNN, count_params
from train_nn   import auto_device, eval_probs, mean_ap, train_one_run, seed_all
from imbalance  import compute_class_weights, make_focal_loss
from eval       import (macro_f1, tune_thresholds_on_val,
                        predict_with_threshold, classification_report)

STRATEGY   = 'A'
NB_ID      = '09'
MODEL_NAME = 'lstm'
SEED       = 42

DEVICE = auto_device()
print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
seed_all(SEED)


Project root: /Users/amrabdullah/AI_2026ss/mlpc
Device:       mps


## Cell 2 — Load splits + mel-spec cache


In [2]:
# ============================================================================
# Cell 2 — Load data, build DataLoaders
# ============================================================================
splits = load_splits(STRATEGY)
print(splits.summary())

cache, meta = load_cache()
print(f"\nMel-spec cache: {cache.shape} {cache.dtype}")
print(f"  global stats: mean={float(meta['global_mean']):.3f} "
      f"std={float(meta['global_std']):.3f}")

# Datasets share the memmap; no extra RAM
ds_train = MelSpecDataset(splits.train_idx, splits.Y_train)
ds_val   = MelSpecDataset(splits.val_idx,   splits.Y_val)
ds_test  = MelSpecDataset(splits.test_idx,  splits.Y_test)

from torch.utils.data import DataLoader
BATCH = 128
N_WORKERS = 0   # memmap + macOS works best with 0 workers

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                      num_workers=N_WORKERS, drop_last=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH * 2, shuffle=False,
                      num_workers=N_WORKERS)
dl_test  = DataLoader(ds_test,  batch_size=BATCH * 2, shuffle=False,
                      num_workers=N_WORKERS)

print(f'\nBatches: train={len(dl_train)}, val={len(dl_val)}, test={len(dl_test)}')


Strategy A
  X_train: (100166, 960)  Y_train: (100166, 15)
  X_val:   (34245, 960)  Y_val:   (34245, 15)
  X_test:  (33828, 960)  Y_test:  (33828, 15)
  D = 960 features, C = 15 classes

Mel-spec cache: (168239, 128, 100) float16
  global stats: mean=4.531 std=2.018

Batches: train=782, val=134, test=133


## Cell 3 — Model factory & sanity check
`fresh_model()` re-seeds before instantiating to guarantee that
each config starts from the same initial weights.


In [3]:
# ============================================================================
# Cell 3 — Model factory
# ============================================================================
def fresh_model():
    seed_all(SEED)
    return CRNN(n_classes=int(splits.C), lstm_hidden=128, lstm_layers=2, dropout=0.4)

_m = fresh_model()
print(f'CRNN params: {count_params(_m):,}')
with torch.no_grad():
    _x = torch.randn(2, 1, 128, 100)
    _y = _m(_x)
print(f'shape check: {tuple(_x.shape)} -> {tuple(_y.shape)}')
del _m, _x, _y


CRNN params: 1,655,503
shape check: (2, 1, 128, 100) -> (2, 15)


## Cell 4 — Config 1: vanilla BCE
Uncalibrated baseline. Early-stop on val mean-AP (threshold-free,
the right metric for under-trained calibrated probabilities — see
MLP v1 retro in `terminal_claude.md`).


In [4]:
# ============================================================================
# Cell 4 — Config 1: vanilla BCE
# ============================================================================
print('=== Config 1: vanilla BCE ===')
m1 = fresh_model().to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(m1.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state1, hist1 = train_one_run(
    m1, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C1 done in {time.time() - t0:.1f}s | {hist1.summary()}')

m1.load_state_dict(state1)
probs_val_c1, Y_val = eval_probs(m1, dl_val, DEVICE)
print(f'C1 val: mAP={mean_ap(probs_val_c1, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c1>=0.5).astype(np.int8)):.4f}")


=== Config 1: vanilla BCE ===


  ep  1  tr_loss=0.1445  val_mAP=0.4223  val_F1@0.5=0.2208  skipped=6


  ep  2  tr_loss=0.1080  val_mAP=0.5061  val_F1@0.5=0.3560  skipped=0


  ep  3  tr_loss=-22951916466110184206368768.0000  val_mAP=0.5210  val_F1@0.5=0.4116  skipped=1


  ep  4  tr_loss=-13897818879528524201328640.0000  val_mAP=0.5437  val_F1@0.5=0.4288  skipped=0


...

  ep 15  tr_loss=0.0539  val_mAP=0.5964  val_F1@0.5=0.5396  skipped=1


  ep 16  tr_loss=-2995702549488886265937920.0000  val_mAP=0.5927  val_F1@0.5=0.5228  skipped=0


  ep 17  tr_loss=-141651881943260.2500  val_mAP=0.5997  val_F1@0.5=0.5439  skipped=1


  ep 18  tr_loss=-8160921206823086551203840.0000  val_mAP=0.5922  val_F1@0.5=0.5552  skipped=1
  early stop at epoch 18 (best=12)
C1 done in 6524.6s | best epoch=12, best val mAP=0.6020, final val F1@0.5=0.5552, final train loss=-8160921206823086551203840.0000


C1 val: mAP=0.6020 F1@0.5=0.5250


## Cell 5 — Config 2: pos-weighted BCE
`sqrt_inverse_freq` clipped to [0.5, 30]. The aggressive
`balanced` weights for `light_switch` (≈370×) destabilize CNN
training; clipping is essential.


In [5]:
# ============================================================================
# Cell 5 — Config 2: pos-weighted BCE (sqrt_inverse_freq)
# ============================================================================
print('=== Config 2: pos-weighted BCE ===')
pw = compute_class_weights(splits.Y_train,
                           kind='sqrt_inverse_freq', clip=(0.5, 30.0))
print(f'pos_weight: min={pw.min():.2f}  max={pw.max():.2f}')

m2 = fresh_model().to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(pw, device=DEVICE))
opt = torch.optim.AdamW(m2.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state2, hist2 = train_one_run(
    m2, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C2 done in {time.time() - t0:.1f}s | {hist2.summary()}')

m2.load_state_dict(state2)
probs_val_c2, _ = eval_probs(m2, dl_val, DEVICE)
print(f'C2 val: mAP={mean_ap(probs_val_c2, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c2>=0.5).astype(np.int8)):.4f}")


=== Config 2: pos-weighted BCE ===
pos_weight: min=2.79  max=19.15


  ep  1  tr_loss=0.3341  val_mAP=0.4658  val_F1@0.5=0.4273  skipped=2


  ep  2  tr_loss=-32995325973710305176322048.0000  val_mAP=0.5298  val_F1@0.5=0.4870  skipped=2


  ep  3  tr_loss=-115209996929367111104462848.0000  val_mAP=0.5190  val_F1@0.5=0.4773  skipped=1


  ep  4  tr_loss=-170515765960529479741734912.0000  val_mAP=0.5512  val_F1@0.5=0.5174  skipped=2


...

  ep 18  tr_loss=-74148955688195216900096000.0000  val_mAP=0.6069  val_F1@0.5=0.5894  skipped=3


  ep 19  tr_loss=0.0982  val_mAP=0.6100  val_F1@0.5=0.5833  skipped=1


  ep 20  tr_loss=0.0930  val_mAP=0.6109  val_F1@0.5=0.5907  skipped=0


  ep 21  tr_loss=-290731238491122.7500  val_mAP=0.6052  val_F1@0.5=0.5803  skipped=4
  early stop at epoch 21 (best=15)
C2 done in 7308.4s | best epoch=15, best val mAP=0.6117, final val F1@0.5=0.5803, final train loss=-290731238491122.7500


C2 val: mAP=0.6117 F1@0.5=0.5844


## Cell 6 — Config 3: focal loss γ=2
Down-weights easy negatives — useful when the negative class
dominates by 100× (the case for `light_switch`).


In [6]:
# ============================================================================
# Cell 6 — Config 3: focal loss γ=2
# ============================================================================
print('=== Config 3: focal loss γ=2 ===')
m3 = fresh_model().to(DEVICE)
loss_fn = make_focal_loss(gamma=2.0, pos_weight=None).to(DEVICE)
opt = torch.optim.AdamW(m3.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state3, hist3 = train_one_run(
    m3, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C3 done in {time.time() - t0:.1f}s | {hist3.summary()}')

m3.load_state_dict(state3)
probs_val_c3, _ = eval_probs(m3, dl_val, DEVICE)
print(f'C3 val: mAP={mean_ap(probs_val_c3, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c3>=0.5).astype(np.int8)):.4f}")


=== Config 3: focal loss γ=2 ===


  ep  1  tr_loss=0.0351  val_mAP=0.4920  val_F1@0.5=0.3247  skipped=4


  ep  2  tr_loss=0.0269  val_mAP=0.5426  val_F1@0.5=0.4152  skipped=2


  ep  3  tr_loss=0.0241  val_mAP=0.5582  val_F1@0.5=0.4495  skipped=0


  ep  4  tr_loss=-9737218098803624036532224.0000  val_mAP=0.5521  val_F1@0.5=0.4511  skipped=3


...

  ep 15  tr_loss=-8363114527832188769009664.0000  val_mAP=0.5998  val_F1@0.5=0.5522  skipped=1


  ep 16  tr_loss=-30780700316566041122570240.0000  val_mAP=0.6004  val_F1@0.5=0.5564  skipped=2


  ep 17  tr_loss=0.0118  val_mAP=0.5956  val_F1@0.5=0.5536  skipped=2


  ep 18  tr_loss=0.0115  val_mAP=0.6016  val_F1@0.5=0.5554  skipped=2
  early stop at epoch 18 (best=12)
C3 done in 6192.6s | best epoch=12, best val mAP=0.6058, final val F1@0.5=0.5554, final train loss=0.0115


C3 val: mAP=0.6058 F1@0.5=0.5329


## Cell 7 — Config 4: best base + per-class threshold tuning
Tie-break by val mAP (threshold-free), then tune per-class
thresholds on val. This isolates the *calibration* effect from
the *learning* effect.


In [7]:
# ============================================================================
# Cell 7 — Pick val-mAP best base, tune per-class thresholds on val
# ============================================================================
configs = {
    'C1': (probs_val_c1, hist1, state1, m1),
    'C2': (probs_val_c2, hist2, state2, m2),
    'C3': (probs_val_c3, hist3, state3, m3),
}
best_key = max(configs, key=lambda k: mean_ap(configs[k][0], Y_val))
print(f'Best base (val mAP): {best_key}')

probs_val_best, _, state_best, model_best = configs[best_key]
class_names = list(splits.class_names)

thr_df = tune_thresholds_on_val(
    probs_val_best, Y_val,
    grid=np.arange(0.05, 0.96, 0.01),
    class_names=class_names)
print(thr_df.round(3))
thresholds = thr_df['threshold'].values
val_macro_f1_c4 = macro_f1(
    Y_val, predict_with_threshold(probs_val_best, thresholds))
print(f'C4 val macro F1 (tuned): {val_macro_f1_c4:.4f}')


Best base (val mAP): C2


                            threshold  val_f1  val_precision  val_recall  \
class                                                                      
bell_ringing                     0.91   0.571          0.677       0.494   
coffee_machine                   0.95   0.681          0.728       0.639   
cutlery_dishes                   0.50   0.634          0.664       0.607   
door_open_close                  0.57   0.433          0.414       0.453   
footsteps                        0.52   0.611          0.564       0.666   
keyboard_typing                  0.76   0.808          0.865       0.757   
keychain                         0.65   0.699          0.694       0.704   
light_switch                     0.74   0.234          0.297       0.193   
microwave                        0.54   0.711          0.747       0.678   
phone_ringing                    0.79   0.721          0.748       0.695   
running_water                    0.53   0.787          0.791       0.784   
toilet_flush

## Cell 8 — Pick overall winner on val macro F1, test once
Test set is touched ONCE per model in this project. No peeking.


In [8]:
# ============================================================================
# Cell 8 — Pick val winner on macro F1, evaluate on test
# ============================================================================
val_macro = {
    'C1_vanilla':         macro_f1(Y_val, (probs_val_c1 >= 0.5).astype(np.int8)),
    'C2_pos_weighted':    macro_f1(Y_val, (probs_val_c2 >= 0.5).astype(np.int8)),
    'C3_focal':           macro_f1(Y_val, (probs_val_c3 >= 0.5).astype(np.int8)),
    f'C4_{best_key}_thr': val_macro_f1_c4,
}
for k, v in val_macro.items():
    print(f'  {k:25s} {v:.4f}')

winner = max(val_macro, key=val_macro.get)
print(f'\nVal winner: {winner}')

if winner.startswith('C4'):
    probs_test, Y_test = eval_probs(model_best, dl_test, DEVICE)
    Y_pred_test = predict_with_threshold(probs_test, thresholds)
else:
    mdl = {'C1_vanilla': m1, 'C2_pos_weighted': m2, 'C3_focal': m3}[winner]
    probs_test, Y_test = eval_probs(mdl, dl_test, DEVICE)
    Y_pred_test = (probs_test >= 0.5).astype(np.int8)

test_report = classification_report(Y_test, Y_pred_test,
                                   class_names=class_names)
test_macro    = float(test_report.loc['macro avg',    'f1'])
test_micro    = float(test_report.loc['micro avg',    'f1'])
test_weighted = float(test_report.loc['weighted avg', 'f1'])
print(f'\nTEST macro F1:    {test_macro:.4f}')
print(f'TEST micro F1:    {test_micro:.4f}')
print(f'TEST weighted F1: {test_weighted:.4f}')


  C1_vanilla                0.5250
  C2_pos_weighted           0.5844
  C3_focal                  0.5329
  C4_C2_thr                 0.6057

Val winner: C4_C2_thr



TEST macro F1:    0.6048
TEST micro F1:    0.6857
TEST weighted F1: 0.6871


## Cell 9 — Plots


In [9]:
# ============================================================================
# Cell 9 — Training curves + per-class F1 + confusion matrix
# ============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

RESULTS = PROJECT_ROOT / 'strategy_A_baseline' / 'artifacts' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Training curves -- one figure per config
for cfg_name, hist in [('c1', hist1), ('c2', hist2), ('c3', hist3)]:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.train_loss)
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('train loss')
    ax[0].set_title(f'09 CRNN {cfg_name.upper()} — train loss')
    ax[1].plot(hist.val_mAP,   label='val mAP')
    ax[1].plot(hist.val_f1_05, label='val F1@0.5')
    ax[1].axvline(hist.best_epoch - 1, ls='--', c='gray', alpha=0.5,
                  label=f'best ep={hist.best_epoch}')
    ax[1].set_xlabel('epoch'); ax[1].legend()
    ax[1].set_title(f'09 CRNN {cfg_name.upper()} — val metrics')
    plt.tight_layout()
    plt.savefig(RESULTS / f'09_lstm_curves_{cfg_name}.png',
                dpi=120, bbox_inches='tight')
    plt.close()

# Per-class F1 bar chart (test)
fig, ax = plt.subplots(figsize=(10, 4))
cls = class_names
ax.bar(range(len(cls)), test_report.loc[cls, 'f1'].values)
ax.set_xticks(range(len(cls))); ax.set_xticklabels(cls, rotation=45, ha='right')
ax.set_ylabel('F1 (test)')
ax.set_title(f'09 CRNN ({winner}) — per-class F1 on test')
plt.tight_layout()
plt.savefig(RESULTS / '09_lstm_per_class_f1.png',
            dpi=120, bbox_inches='tight')
plt.close()

# Confusion matrix (argmax single-label)
Y_t = Y_test.argmax(axis=1); Y_p = Y_pred_test.argmax(axis=1)
cm = confusion_matrix(Y_t, Y_p, labels=range(splits.C))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(splits.C)); ax.set_yticks(range(splits.C))
ax.set_xticklabels(cls, rotation=45, ha='right'); ax.set_yticklabels(cls)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'09 CRNN ({winner}) — confusion (argmax)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(RESULTS / '09_lstm_confusion.png',
            dpi=120, bbox_inches='tight')
plt.close()

print(f'plots -> {RESULTS}')


plots -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results


## Cell 10 — Save results JSON


In [10]:
# ============================================================================
# Cell 10 — Save 08_cnn.json (schema mirrors 07_mlp.json)
# ============================================================================
result = {
    'meta': {
        'strategy':    STRATEGY,
        'model':       MODEL_NAME,
        'notebook_id': NB_ID,
        'n_train':     int(splits.N_train),
        'n_val':       int(splits.N_val),
        'n_test':      int(splits.N_test),
        'n_params':    int(count_params(m1)),
        'device':      str(DEVICE),
        'random_seed': SEED,
    },
    'val_scores_macro_f1': val_macro,
    'val_scores_mAP': {
        'C1': float(mean_ap(probs_val_c1, Y_val)),
        'C2': float(mean_ap(probs_val_c2, Y_val)),
        'C3': float(mean_ap(probs_val_c3, Y_val)),
    },
    'histories': {
        'C1': {'train_loss': hist1.train_loss, 'val_mAP': hist1.val_mAP,
               'val_f1_05': hist1.val_f1_05, 'best_epoch': hist1.best_epoch},
        'C2': {'train_loss': hist2.train_loss, 'val_mAP': hist2.val_mAP,
               'val_f1_05': hist2.val_f1_05, 'best_epoch': hist2.best_epoch},
        'C3': {'train_loss': hist3.train_loss, 'val_mAP': hist3.val_mAP,
               'val_f1_05': hist3.val_f1_05, 'best_epoch': hist3.best_epoch},
    },
    'thresholds_c4': {c: float(t) for c, t in zip(class_names, thresholds)},
    'winning_config': {
        'name': winner,
        'thresholds': [float(t) for t in thresholds]
                      if winner.startswith('C4')
                      else [0.5] * int(splits.C),
    },
    'test_report': test_report.to_dict(),
    'headline_test': {
        'macro_f1':    test_macro,
        'micro_f1':    test_micro,
        'weighted_f1': test_weighted,
    },
}

out_path = RESULTS / f'{NB_ID}_{MODEL_NAME}.json'
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2, default=float)
print(f'saved -> {out_path}')


saved -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results/09_lstm.json
